# Chapter 6: AI and Algorithmic Trading Strategies

Companion notebook for Chapter 6 of *Mastering AI-Powered Finance* (`Overleaf/Book 1`). It holds the generative-AI part of the trading tutorial from the full manuscript's Chapter 6; the source references below point to that manuscript.

It builds one component: a daily news-sentiment signal produced by a large language model. The Chapter 7 notebook builds and backtests the ML trading system on SPY to which this signal can be added as a feature.

Run the notebook from top to bottom. The classifier that calls the Anthropic API is shown as code, not executed, so the notebook runs offline and without credentials. To try it live, copy it into a code cell and set `ANTHROPIC_API_KEY`.

In [1]:
import numpy as np
import pandas as pd

## AI Tutorial: An LLM News-Sentiment Signal

### Code example 1
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

```python
import anthropic
import json
import os
import math

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env variable

def classify_sentiment(headlines: list[str]) -> list[dict]:
    """Classify financial headlines using Claude as a zero-shot sentiment model."""
    prompt = f"""Classify each financial headline as BULLISH, BEARISH, or NEUTRAL.
Return ONLY a JSON array with objects containing "headline", "sentiment",
and "confidence" (0-1). Headlines:
{chr(10).join(f'{i+1}. {h}' for i, h in enumerate(headlines))}"""

    response = client.messages.create(
        model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5-5"),
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    if response.stop_reason != 'end_turn':
        raise ValueError('Sentiment response was incomplete; do not trade on it.')
    raw = ''.join(block.text for block in response.content if block.type == 'text').strip()
    if raw.startswith('```') and raw.endswith('```'):
        raw = '\n'.join(raw.splitlines()[1:-1])
    records = json.loads(raw)
    if not isinstance(records, list) or len(records) != len(headlines):
        raise ValueError('Expected one classification per input headline.')
    for headline, record in zip(headlines, records):
        if not isinstance(record, dict) or record.get('headline') != headline:
            raise ValueError('Response headlines must match inputs in order.')
        confidence = record.get('confidence')
        if (record.get('sentiment') not in {'BULLISH', 'BEARISH', 'NEUTRAL'}
                or isinstance(confidence, bool)
                or not isinstance(confidence, (int, float))
                or not math.isfinite(confidence) or not 0 <= confidence <= 1):
            raise ValueError('Invalid sentiment label or confidence.')
    return records

# Example: classify today's headlines for SPY-relevant news
headlines = [
    "Fed signals potential rate cut amid cooling inflation data",
    "Major tech earnings disappoint, Nasdaq futures drop 2%",
    "US manufacturing PMI holds steady at 51.2 in October",
    "Oil prices surge after OPEC announces surprise production cut",
    "Consumer confidence index rises to highest level since 2021"
]

sentiment_results = classify_sentiment(headlines)
for r in sentiment_results:
    print(f"  [{r['sentiment']:>7s}] (conf: {r['confidence']:.2f})  {r['headline']}")
```

Optional live LLM sentiment example. It requires `ANTHROPIC_API_KEY`, so it is preserved as a listing rather than executed in the default notebook run.

Model-generated confidence is an uncalibrated score, not a probability of a profitable trade. This example does not establish historical availability of the model, news, or classifications.

### Manuscript illustration (not a verified execution result)
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

```text
  [BULLISH] (conf: 0.85)  Fed signals potential rate cut amid cooling inflation
  [BEARISH] (conf: 0.90)  Major tech earnings disappoint, Nasdaq futures drop 2%
  [NEUTRAL] (conf: 0.80)  US manufacturing PMI holds steady at 51.2 in October
  [BEARISH] (conf: 0.65)  Oil prices surge after OPEC announces surprise production cut
  [BULLISH] (conf: 0.80)  Consumer confidence index rises to highest level since 2021
```

### Code example 2
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

In [2]:
def daily_sentiment_signal(headlines_with_timestamps, date, as_of):
    """Use only same-day headlines actually available by an aware cutoff.

    Each record needs timestamp (publication) and available_at (when its
    classification became usable), both with time zones. date is an exchange
    calendar date; as_of is the actual after-close decision timestamp.
    """
    cutoff = pd.Timestamp(as_of)
    if cutoff.tzinfo is None:
        raise ValueError('as_of must include a time zone.')
    session_date = pd.Timestamp(date).date()
    if cutoff.tz_convert('America/New_York').date() != session_date:
        raise ValueError('date must match the New York decision date.')
    sentiment_map = {'BULLISH': 1.0, 'NEUTRAL': 0.0, 'BEARISH': -1.0}
    scores = []
    for h in headlines_with_timestamps:
        published, available = pd.Timestamp(h['timestamp']), pd.Timestamp(h['available_at'])
        if published.tzinfo is None or available.tzinfo is None:
            raise ValueError('Headline timestamps must include time zones.')
        if available < published:
            raise ValueError('Classification availability cannot precede publication.')
        if published.tz_convert('America/New_York').date() != session_date or available > cutoff:
            continue
        confidence = float(h['confidence'])
        if not np.isfinite(confidence) or not 0 <= confidence <= 1:
            raise ValueError('confidence must be finite and in [0, 1].')
        scores.append(sentiment_map[h['sentiment']] * confidence)
    return float(np.mean(scores)) if scores else 0.0  # imputed neutral

# Availability includes ingestion and classification delay, not just publication.
sample_news = [
    {'timestamp': '2024-10-01T15:00:00-04:00',
     'available_at': '2024-10-01T15:01:00-04:00',
     'sentiment': 'BULLISH', 'confidence': 0.8},
    {'timestamp': '2024-10-01T17:00:00-04:00',
     'available_at': '2024-10-01T17:01:00-04:00',
     'sentiment': 'BEARISH', 'confidence': 0.9},
]
print(daily_sentiment_signal(sample_news, '2024-10-01', '2024-10-01T16:05:00-04:00'))

0.8
